# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


In [2]:
# # 1. 필수 라이브러리 설치 (버전 고정)
# !pip -q install git+https://github.com/huggingface/transformers accelerate
# !pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
# !pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets

# # 2. 충돌 방지를 위한 pandas 및 pillow 버전 강제 지정
# !pip install pandas==2.2.2 "pillow<12.0"

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [6]:
# # 구글드라이브 마운트
# from google.colab import drive
# drive.mount('/content/drive')
# # 압축 해제
# !unzip "/content/drive/My Drive/060401-ai-challenge.zip" -d "/content/"

# 라이브러리, 데이터, 설정

In [5]:
import os, random, math, json, gc, re
from dataclasses import dataclass
from contextlib import nullcontext
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

Image.MAX_IMAGE_PIXELS = None

DATA_ROOT  = "/content"
TRAIN_CSV  = os.path.join(DATA_ROOT, "train.csv")
TEST_CSV   = os.path.join(DATA_ROOT, "test.csv")
AUDIT_DIR  = os.path.join(DATA_ROOT, "audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 2
SEED       = 42
VAL_SIZE   = 0.10
BATCH_SIZE = 4        # 버그픽스: 1 → 4
GRAD_ACCUM = 2        # 버그픽스: 8 → 2 (effective batch 8 유지)
LR         = 1e-4
NUM_EPOCHS = 3        # 버그픽스: 1 → 5
EARLY_STOP_PATIENCE = 2
USE_4BIT   = False    # 버그픽스: H100이므로 False
CHECK_IMAGES = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

def resolve_path(p):
    p = str(p)
    return p if os.path.isabs(p) else os.path.join(DATA_ROOT, p)

train_df["abs_path"] = train_df["path"].map(resolve_path)
test_df["abs_path"]  = test_df["path"].map(resolve_path)

def find_hard_errors(df, has_answer=True, check_images=True):
    issue_frames = []
    base_cols = ["id","path","abs_path","question","a","b","c","d"]
    if has_answer:
        base_cols.append("answer")

    # 1) 결측치
    missing_mask = df[base_cols].isna().any(axis=1)
    if missing_mask.any():
        tmp = df.loc[missing_mask, base_cols].copy()
        tmp["issue"] = "missing_value"
        issue_frames.append(tmp)

    # 2) 유효하지 않은 answer
    if has_answer:
        invalid_mask = ~df["answer"].astype(str).str.strip().str.lower().isin(list("abcd"))
        if invalid_mask.any():
            tmp = df.loc[invalid_mask, base_cols].copy()
            tmp["issue"] = "invalid_answer"
            issue_frames.append(tmp)

    # 3) 보기 중복 (train_4196 등)
    dup_mask = df[["a","b","c","d"]].astype(str).apply(
        lambda row: len({x.strip() for x in row.tolist()}) < 4, axis=1
    )
    if dup_mask.any():
        tmp = df.loc[dup_mask, base_cols].copy()
        tmp["issue"] = "duplicate_options"
        issue_frames.append(tmp)

    # 4) 파일 없음
    no_file_mask = ~df["abs_path"].map(os.path.exists)
    if no_file_mask.any():
        tmp = df.loc[no_file_mask, base_cols].copy()
        tmp["issue"] = "missing_file"
        issue_frames.append(tmp)

    # 5) 깨진 이미지
    if check_images:
        broken = []
        for idx in tqdm(df.index[~no_file_mask].tolist(), desc="Image audit"):
            try:
                with Image.open(df.at[idx, "abs_path"]) as img:
                    img.verify()
            except Exception:
                broken.append(idx)
        if broken:
            tmp = df.loc[broken, base_cols].copy()
            tmp["issue"] = "broken_image"
            issue_frames.append(tmp)

    if issue_frames:
        issues = pd.concat(issue_frames, ignore_index=True).drop_duplicates(subset=["id","issue"])
    else:
        issues = pd.DataFrame(columns=base_cols + ["issue"])

    bad_ids  = set(issues["id"].tolist())
    clean_df = df[~df["id"].isin(bad_ids)].copy().reset_index(drop=True)
    return clean_df, issues

train_clean_df, train_issue_df = find_hard_errors(
    train_df, has_answer=True, check_images=CHECK_IMAGES
)

train_issue_df.to_csv(os.path.join(AUDIT_DIR, "train_hard_errors.csv"), index=False)

audit_summary = {
    "raw": int(len(train_df)),
    "clean": int(len(train_clean_df)),
    "removed": int(len(train_df) - len(train_clean_df)),
    "issues": train_issue_df["issue"].value_counts().to_dict() if len(train_issue_df) else {},
    "answer_dist": train_clean_df["answer"].value_counts().sort_index().to_dict(),
}
print(json.dumps(audit_summary, ensure_ascii=False, indent=2))

# === 의심 샘플 제거 (나중에 채우기) ===
REMOVE_IDS = []  # ex: ["train_0123", "train_0456"]
if REMOVE_IDS:
    train_clean_df = train_clean_df[~train_clean_df["id"].isin(REMOVE_IDS)].reset_index(drop=True)
    print(f"제거 후: {len(train_clean_df)}개")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(train_clean_df, train_clean_df["answer"]))

print(f"K-Fold: {N_FOLDS}개 fold")
for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"  Fold {fold_idx}: train={len(train_idx)}, val={len(val_idx)}")

Device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv'

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# Processor만 로드 (모델은 K-fold 루프에서 매번 새로 로드)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Processor 로드 완료")

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 재활용품 이미지 분석 전문가입니다. "
    "이미지에서 색상, 재질, 개수, 형태를 주의 깊게 관찰하세요. "
    "주어진 선택지 중 이미지와 가장 정확히 일치하는 하나를 고르세요. "
    "반드시 a, b, c, d 중 하나의 소문자 알파벳만 출력하세요. 다른 설명은 절대 금지입니다."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "정답:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.train     = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["abs_path"]).convert("RGB")

        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        gold = str(row["answer"]).strip().lower() if "answer" in row else None

        user_text = build_mc_prompt(str(row["question"]), a, b, c, d)

        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]

        item = {
            "prompt_messages": prompt_messages,
            "image": img,
            "gold":  gold,
        }

        if self.train:
            item["full_messages"] = prompt_messages + [
                {"role": "assistant",
                 "content": [{"type": "text", "text": gold}]}
            ]

        return item


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        full_texts, prompt_texts, images = [], [], []

        for sample in batch:
            full_texts.append(
                self.processor.apply_chat_template(
                    sample["full_messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )
            )
            prompt_texts.append(
                self.processor.apply_chat_template(
                    sample["prompt_messages"],
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )
            images.append(sample["image"])

        enc = self.processor(
            text=full_texts, images=images,
            padding=True, return_tensors="pt",
        )
        prompt_enc = self.processor(
            text=prompt_texts, images=images,
            padding=True, return_tensors="pt",
        )

        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        prompt_lens = prompt_enc["attention_mask"].sum(dim=1).tolist()
        for i, pl in enumerate(prompt_lens):
            labels[i, :pl] = -100

        enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
def extract_choice(text: str) -> str:
    text = str(text).strip().lower()
    if text in {"a","b","c","d"}:
        return text
    match = re.findall(r"[abcd]", text)
    return match[0] if match else "a"

@torch.no_grad()
def predict_one(model, processor, row, device):
    img = Image.open(row["abs_path"]).convert("RGB")
    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]),
        str(row["c"]), str(row["d"]),
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": user_text},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    amp_ctx = torch.cuda.amp.autocast(dtype=torch.float16) if device == "cuda" else nullcontext()
    with amp_ctx:
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    gen_ids   = out_ids[:, inputs["input_ids"].shape[1]:]
    pred_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return extract_choice(pred_text), pred_text

@torch.no_grad()
def evaluate_generation_accuracy(model, processor, valid_df, device):
    refs, preds, raws = [], [], []
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="valid-acc"):
        pred, raw = predict_one(model, processor, row, device)
        preds.append(pred)
        raws.append(raw)
        refs.append(str(row["answer"]).strip().lower())
    acc = float(np.mean(np.array(preds) == np.array(refs)))
    return acc, preds, raws, refs

# === K-Fold 학습 ===
DRIVE_ROOT = "/content/drive/My Drive/kfold_models"
os.makedirs(DRIVE_ROOT, exist_ok=True)

all_fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"{'='*60}")

    train_subset = train_clean_df.iloc[train_idx].reset_index(drop=True)
    valid_subset = train_clean_df.iloc[val_idx].reset_index(drop=True)

    # 모델 새로 로드
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    ).to(device)

    base_model.gradient_checkpointing_enable({"use_reentrant": False})
    base_model.config.use_cache = False

    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)

    # DataLoader
    train_ds = VQAMCDataset(train_subset, processor, train=True)
    valid_ds = VQAMCDataset(valid_subset, processor, train=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, collate_fn=DataCollator(processor, True),
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, collate_fn=DataCollator(processor, True),
    )

    # 옵티마이저
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    num_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(num_update_steps * 0.03)),
        num_training_steps=num_update_steps,
    )

    # 학습
    best_val_acc = -1.0
    best_epoch = 0
    patience_counter = 0
    SAVE_DIR = os.path.join(DRIVE_ROOT, f"fold_{fold_idx}")
    os.makedirs(SAVE_DIR, exist_ok=True)

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_loss_sum, train_steps = 0.0, 0

        pbar = tqdm(train_loader, desc=f"Fold{fold_idx} Ep{epoch} [train]")
        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                loss = model(**batch).loss / GRAD_ACCUM
            loss.backward()
            train_loss_sum += loss.item() * GRAD_ACCUM
            train_steps += 1

            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{train_loss_sum / train_steps:.4f}")

        train_loss = train_loss_sum / max(1, train_steps)

        # Valid
        model.eval()
        val_acc, val_preds, val_raws, val_refs = evaluate_generation_accuracy(
            model, processor, valid_subset, device
        )

        print(f"  Fold{fold_idx} Ep{epoch}: loss={train_loss:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            model.save_pretrained(SAVE_DIR)
            processor.save_pretrained(SAVE_DIR)
            print(f"    ✅ best 저장 (val_acc={best_val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"    🛑 Early stop")
                break

    all_fold_results.append({
        "fold": fold_idx, "best_epoch": best_epoch, "best_val_acc": best_val_acc
    })

    # 메모리 해제
    del model, base_model, optimizer, scheduler
    del train_ds, valid_ds, train_loader, valid_loader
    gc.collect()
    torch.cuda.empty_cache()

# 결과 요약
print(f"\n{'='*60}")
print("K-Fold 결과 요약")
print(f"{'='*60}")
results_df = pd.DataFrame(all_fold_results)
print(results_df)
print(f"\n평균 val_acc: {results_df['best_val_acc'].mean():.4f}")
results_df.to_csv(os.path.join(DRIVE_ROOT, "kfold_results.csv"), index=False)

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from collections import Counter

# === K-Fold + TTA 앙상블 추론 ===
TTA_SIZES = [384, 448, 512]  # 해상도 3개
all_test_preds = []  # (fold × tta) 개의 예측 리스트

for fold_idx in range(N_FOLDS):
    SAVE_DIR = os.path.join(DRIVE_ROOT, f"fold_{fold_idx}")

    # 모델 로드
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True,
    ).to(device)
    from peft import PeftModel
    model = PeftModel.from_pretrained(base_model, SAVE_DIR).to(device)
    model.eval()

    for tta_size in TTA_SIZES:
        print(f"\nFold {fold_idx} / TTA {tta_size} 추론 중...")

        # 해상도별 processor 생성
        tta_processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            min_pixels=tta_size * tta_size,
            max_pixels=tta_size * tta_size,
            trust_remote_code=True,
        )
        if tta_processor.tokenizer.pad_token is None:
            tta_processor.tokenizer.pad_token = tta_processor.tokenizer.eos_token

        fold_preds = []
        for i in tqdm(range(len(test_df)), desc=f"F{fold_idx}_TTA{tta_size}"):
            row = test_df.iloc[i]
            pred, _ = predict_one(model, tta_processor, row, device)
            fold_preds.append(pred)

        all_test_preds.append(fold_preds)

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

# 다수결 투표 (5 fold × 3 TTA = 15표)
print(f"\n총 {len(all_test_preds)}개 예측으로 다수결 투표")
final_preds = []
for i in range(len(test_df)):
    votes = [preds[i] for preds in all_test_preds]
    winner = Counter(votes).most_common(1)[0][0]
    final_preds.append(winner)

# 제출 파일
submission = pd.DataFrame({"id": test_df["id"], "answer": final_preds})
submission.to_csv("/content/drive/My Drive/submission_kfold_tta.csv", index=False)
print(f"저장 완료: submission_kfold_tta.csv")
print(f"투표 구성: {N_FOLDS} folds × {len(TTA_SIZES)} TTA = {len(all_test_preds)}표")

In [ ]:
# 모델 응답 예시
print(output_text)